# 03 — Product Score Index (PSI)

Este notebook calcula o PSI e exporta rankings gerais e por categoria para `reports/`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from src.scoring import PSIWeights, add_psi_column


C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Carregar base de produtos

In [2]:
df_products = pd.read_csv(PROCESSED_DIR / 'base_produtos.csv')
df_products.shape


(1351, 17)

## Calcular PSI

In [3]:
weights = PSIWeights(rating=0.40, rating_count_log=0.35, discount_pct=0.25)
df_products_psi = add_psi_column(df_products, weights=weights)
df_products_psi[['product_name','main_category','rating_clean','rating_count_clean','discount_pct_clean','PSI']].head(5)


,product_name,main_category,rating_clean,rating_count_clean,discount_pct_clean,PSI
0,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,Computers&Accessories,4.1,8131.0,58.0,66.742266
1,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,Computers&Accessories,4.2,179692.0,44.0,73.482629
2,Duracell Plus AAA Rechargeable Batteries (750 ...,Electronics,4.3,27201.0,20.0,62.864188
3,"Logitech B100 Wired USB Mouse, 3 yr Warranty, ...",Computers&Accessories,4.3,31534.0,26.0,64.895912
4,"Logitech M235 Wireless Mouse, 1000 DPI Optical...",Computers&Accessories,4.5,54405.0,30.0,70.235071


## Rankings e exportações

In [4]:
df_products_psi = df_products_psi.sort_values('PSI', ascending=False).reset_index(drop=True)
top10 = df_products_psi.head(10)
bottom10 = df_products_psi.tail(10)
top10[['product_id','product_name','main_category','discounted_price_clean','discount_pct_clean','rating_clean','rating_count_clean','PSI']]


,product_id,product_name,main_category,discounted_price_clean,discount_pct_clean,rating_clean,rating_count_clean,PSI
0,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",Electronics,309.0,78.0,4.4,426973.0,87.744681
1,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,Electronics,219.0,69.0,4.4,426973.0,85.351064
2,B00NH11KIK,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,Computers&Accessories,209.0,70.0,4.5,107687.0,82.887274
3,B00NH11PEY,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,Computers&Accessories,199.0,73.0,4.5,74976.0,82.617210
4,B07Q4QV1DL,ELV Aluminum Adjustable Mobile Phone Foldable ...,Electronics,269.0,82.0,4.5,28978.0,82.206872
5,B088ZFJY82,Elv Aluminium Adjustable Mobile Phone Foldable...,Electronics,314.0,79.0,4.5,28978.0,81.409000
6,B09MT84WV5,Samsung EVO Plus 128GB microSDXC UHS-I U3 130M...,Electronics,1149.0,71.0,4.3,140036.0,81.261341
7,B08L5FM4JC,"SanDisk Ultra microSD UHS-I Card 64GB, 120MB/s R",Electronics,649.0,73.0,4.4,67260.0,80.963544
8,B09MT6XSFW,Samsung EVO Plus 64GB microSDXC UHS-I U1 130MB...,Electronics,599.0,68.0,4.3,140036.0,80.463469
9,B07GXHC691,STRIFF PS2_01 Multi Angle Mobile/Tablet Tablet...,Electronics,99.0,80.0,4.3,42641.0,80.147642


In [5]:
df_products_psi.to_csv(PROCESSED_DIR / 'base_produtos_psi.csv', index=False)
top10.to_csv(REPORTS_DIR / 'psi_top10_overall.csv', index=False)
bottom10.to_csv(REPORTS_DIR / 'psi_bottom10_overall.csv', index=False)

top10_by_cat = (
    df_products_psi
    .sort_values(['main_category','PSI'], ascending=[True, False])
    .groupby('main_category', as_index=False)
    .head(10)
)
top10_by_cat.to_csv(REPORTS_DIR / 'psi_top10_by_category.csv', index=False)
(PROCESSED_DIR / 'base_produtos_psi.csv', REPORTS_DIR / 'psi_top10_by_category.csv')


(WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/data/processed/base_produtos_psi.csv'),
 WindowsPath('C:/Users/flavi/Documents/GitHub/Amazon_Product_Clustering/amazon-product-intelligence/reports/psi_top10_by_category.csv'))